# 19 — GLOBE Patch Analysis

Analysis of the 224×224 patches extracted from NASA GLOBE images in notebook 18.
The extraction pipeline is reproduced here unchanged; analysis cells follow the capping step.

**Questions answered here:**
- What are the source image resolutions, and how many patches does each image produce?
- How do cloud-filter confidence scores distribute per class?
- What do per-class pixel statistics (brightness, colour) look like?
- What does the mean patch for each class look like?
- How similar / separable are the classes in pixel space (PCA)?

## Setup — data indexing (from nb18)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from collections import Counter

GCD_ROOT         = Path('../resources/cloud-images/GCD')
GCD_ONLY_CLASSES = {'4_clearsky', '7_mixed'}

def index_gcd_split(split_dir):
    paths, labels = [], []
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        label = class_dir.name
        if label not in GCD_ONLY_CLASSES:
            continue
        for img_path in sorted(class_dir.glob('*.jpg')):
            paths.append(str(img_path))
            labels.append(label)
    return np.array(paths), np.array(labels)

gcd_train_paths_raw, gcd_train_labels_raw = index_gcd_split(GCD_ROOT / 'train')
gcd_test_paths_raw,  gcd_test_labels_raw  = index_gcd_split(GCD_ROOT / 'test')

print('GCD (clearsky + mixed only):')
for split_name, lbl in [('train', gcd_train_labels_raw), ('test', gcd_test_labels_raw)]:
    print(f'  {split_name}:')
    for cls, cnt in sorted(Counter(lbl).items()):
        print(f'    {cls:25s}  {cnt:,}')
    print(f'    {"TOTAL":25s}  {len(lbl):,}')

GCD (clearsky + mixed only):
  train:
    4_clearsky                 2,150
    7_mixed                    348
    TOTAL                      2,498
  test:
    4_clearsky                 1,589
    7_mixed                    607
    TOTAL                      2,196


In [2]:
CSV_PATH    = Path('../resources/cloud-images/NASA_GLOBE_CD/cloud_filter_results_1.csv')
IMAGES_ROOT = Path('../resources/cloud-images/NASA_GLOBE_CD/downloaded_images')

GLOBE_TO_GCD = {
    'Ac': '2_altocumulus',
    'Cb': '6_cumulonimbus',
    'Ci': '3_cirrus',
    'Cu': '1_cumulus',
    'Sc': '5_stratocumulus',
}

def index_globe_filtered(csv_path, globe_to_gcd, images_root,
                          conf_threshold=0.99, max_per_class=3000):
    df = pd.read_csv(csv_path)
    df = df.rename(columns={'cloud_conf': 'head2_conf', 'head1_class': 'head1_pred'})
    paths, labels = [], []
    for globe_cls, gcd_label in globe_to_gcd.items():
        clean = df[(df['folder'] == globe_cls) & (df['head2_conf'] >= conf_threshold)]
        clean = clean.sample(min(max_per_class, len(clean)), random_state=42)
        for _, row in clean.iterrows():
            paths.append(str(images_root / globe_cls / row['filename']))
            labels.append(gcd_label)
    return np.array(paths), np.array(labels)

globe_paths, globe_labels = index_globe_filtered(CSV_PATH, GLOBE_TO_GCD, IMAGES_ROOT)

print('NASA GLOBE (5 matching classes, conf >= 0.99):')
for cls, cnt in sorted(Counter(globe_labels).items()):
    print(f'  {cls:25s}  {cnt:,}')
print(f'  {"TOTAL":25s}  {len(globe_labels):,}')

NASA GLOBE (5 matching classes, conf >= 0.99):
  1_cumulus                  3,000
  2_altocumulus              3,000
  3_cirrus                   3,000
  5_stratocumulus            3,000
  6_cumulonimbus             3,000
  TOTAL                      15,000


In [3]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'device: {device}')

device: mps


In [4]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedShuffleSplit

ALL_CLASSES = sorted(GCD_ONLY_CLASSES | set(GLOBE_TO_GCD.values()))
le = LabelEncoder()
le.fit(ALL_CLASSES)
class_names = le.classes_
n_classes   = len(class_names)
print(f'Classes ({n_classes}): {class_names}')

globe_enc = le.transform(globe_labels)

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(sss1.split(globe_paths, globe_enc))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1 / 0.8, random_state=42)
train_rel_idx, val_rel_idx = next(
    sss2.split(globe_paths[trainval_idx], globe_enc[trainval_idx]))

globe_split = {
    'train': (globe_paths[trainval_idx[train_rel_idx]], globe_enc[trainval_idx[train_rel_idx]]),
    'val':   (globe_paths[trainval_idx[val_rel_idx]],   globe_enc[trainval_idx[val_rel_idx]]),
    'test':  (globe_paths[test_idx],                    globe_enc[test_idx]),
}

print('\nGLOBE image counts (before patch expansion):')
for split_name, (paths, _) in globe_split.items():
    print(f'  {split_name:5s}: {len(paths):,}')

Classes (7): ['1_cumulus' '2_altocumulus' '3_cirrus' '4_clearsky' '5_stratocumulus'
 '6_cumulonimbus' '7_mixed']

GLOBE image counts (before patch expansion):
  train: 10,500
  val  : 1,500
  test : 3,000


## Patch extraction (from nb18)

In [5]:
from torch.utils.data import Dataset
from PIL import Image

CROP_FRACTION = 0.5
OVERLAP       = 0.25
OUTPUT_SIZE   = 224


class PatchDataset(Dataset):
    """
    Expands each image into proportional 224x224 patches.
    Crop boxes are computed once at init (reads only image headers).
    Pixel data is loaded lazily in __getitem__.
    Each patch inherits its parent image's label.
    """
    def __init__(self, paths, encoded_labels,
                 crop_fraction=0.5, overlap=0.25, output_size=224,
                 transform=None):
        self.output_size = output_size
        self.transform   = transform
        self.samples     = []  # (path, label_int, (x0, y0, x1, y1))

        for path, label in zip(paths, encoded_labels):
            try:
                W, H = Image.open(path).size
            except Exception:
                continue
            crop_size = max(output_size, int(min(W, H) * crop_fraction))
            stride    = int(crop_size * (1 - overlap))
            for y in range(0, H - crop_size + 1, stride):
                for x in range(0, W - crop_size + 1, stride):
                    self.samples.append((path, label, (x, y, x + crop_size, y + crop_size)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, box = self.samples[idx]
        img  = Image.open(path).convert('RGB')
        crop = img.crop(box).resize((self.output_size, self.output_size), Image.BILINEAR)
        if self.transform:
            crop = self.transform(crop)
        return crop, label


print('Building GLOBE patch datasets (reading image headers)...')
globe_train_patches = PatchDataset(
    globe_split['train'][0], globe_split['train'][1],
    CROP_FRACTION, OVERLAP, OUTPUT_SIZE)
globe_val_patches   = PatchDataset(
    globe_split['val'][0],   globe_split['val'][1],
    CROP_FRACTION, OVERLAP, OUTPUT_SIZE)
globe_test_patches  = PatchDataset(
    globe_split['test'][0],  globe_split['test'][1],
    CROP_FRACTION, OVERLAP, OUTPUT_SIZE)

print(f'GLOBE patches — train: {len(globe_train_patches):,}  '
      f'val: {len(globe_val_patches):,}  test: {len(globe_test_patches):,}')
avg = len(globe_train_patches) / len(globe_split['train'][0])
print(f'Avg patches per training image: {avg:.1f}')

Building GLOBE patch datasets (reading image headers)...
GLOBE patches — train: 69,196  val: 9,902  test: 19,810
Avg patches per training image: 6.6


In [6]:
import torchvision
import torch.nn as nn
import torchvision.transforms as T_filter
import torch.nn.functional as F_filter

CLOUD_THRESHOLD = 0.99
FILTER_BATCH    = 128


class MultiHeadResNet18(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.head1 = base.fc
        self.head2 = nn.Linear(512, num_classes)
        for p in self.backbone.parameters(): p.requires_grad = False
        for p in self.head1.parameters():    p.requires_grad = False

    def forward(self, x):
        feat = self.backbone(x).flatten(1)
        return self.head1(feat), self.head2(feat)


filter_ckpt    = torch.load('../models/multihead_resnet18_cloud_sky.pth',
                            map_location='cpu', weights_only=False)
filter_classes = filter_ckpt['classes']
cloud_idx      = filter_classes.index('cloud')
filter_model   = MultiHeadResNet18(num_classes=len(filter_classes))
filter_model.load_state_dict(filter_ckpt['model_state_dict'])
filter_model.eval().to(device)

filter_transform = T_filter.Compose([
    T_filter.ToTensor(),
    T_filter.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def filter_patch_samples(samples, threshold=CLOUD_THRESHOLD, batch_size=FILTER_BATCH):
    """Returns (kept_samples, all_confidences) so we can plot confidence distributions."""
    kept   = []
    confs  = []  # (sample, cloud_conf) for ALL patches before threshold
    tensor_buf, sample_buf = [], []
    current_path, current_img = None, None

    def flush():
        if not tensor_buf:
            return
        inp = torch.stack(tensor_buf).to(device)
        with torch.no_grad():
            _, out2 = filter_model(inp)
            probs = F_filter.softmax(out2, dim=1).cpu().numpy()
        for samp, prob in zip(sample_buf, probs):
            conf = prob[cloud_idx]
            confs.append((samp, conf))
            if conf >= threshold:
                kept.append(samp)
        tensor_buf.clear()
        sample_buf.clear()

    for sample in samples:
        path, label, box = sample
        if path != current_path:
            try:
                current_img  = Image.open(path).convert('RGB')
                current_path = path
            except Exception:
                current_img  = None
                current_path = path
        if current_img is None:
            continue
        try:
            crop = current_img.crop(box).resize((OUTPUT_SIZE, OUTPUT_SIZE), Image.BILINEAR)
            tensor_buf.append(filter_transform(crop))
            sample_buf.append(sample)
        except Exception:
            pass
        if len(tensor_buf) >= batch_size:
            flush()

    flush()
    return kept, confs


print(f'Running patch cloud filter  threshold={CLOUD_THRESHOLD}  (train split only)...')
train_kept, train_confs = filter_patch_samples(globe_train_patches.samples)
globe_train_patches.samples = train_kept

print(f'train: {len(train_confs):,} patches scored  →  {len(train_kept):,} kept  '
      f'({100*len(train_kept)/max(len(train_confs),1):.1f}%)')

Running patch cloud filter  threshold=0.99  (train split only)...


KeyboardInterrupt: 

In [ ]:
import random as _random

MAX_PATCHES_PER_CLASS = 3000
_rng = _random.Random(42)

by_class = {}
for sample in globe_train_patches.samples:
    by_class.setdefault(sample[1], []).append(sample)

capped = []
print(f'Capping GLOBE training patches to {MAX_PATCHES_PER_CLASS:,}/class (seed=42)\n')
for cls_idx in sorted(by_class):
    pool   = by_class[cls_idx]
    chosen = _rng.sample(pool, min(MAX_PATCHES_PER_CLASS, len(pool)))
    capped.extend(chosen)
    print(f'  {class_names[cls_idx]:25s}  {len(pool):>7,} → {len(chosen):>5,}')

globe_train_patches.samples = capped
print(f'\n  GLOBE train patches total: {len(globe_train_patches.samples):,}')

---
## Analysis

### 1. Source image resolutions

In [ ]:
# Collect (width, height) for every unique source image in the train split
seen = set()
res_records = []  # (w, h, label_idx)

for path, label, _ in globe_train_patches.samples:
    if path in seen:
        continue
    seen.add(path)
    try:
        W, H = Image.open(path).size
        res_records.append({'path': path, 'w': W, 'h': H,
                            'mp': W * H / 1e6, 'class': class_names[label]})
    except Exception:
        pass

res_df = pd.DataFrame(res_records)
print(f'Unique source images: {len(res_df):,}')
print(res_df[['w', 'h', 'mp']].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Source image resolution distribution (train split)', fontsize=12)

for ax, col, label in zip(
    axes,
    ['w', 'h', 'mp'],
    ['Width (px)', 'Height (px)', 'Megapixels']
):
    for cls in sorted(res_df['class'].unique()):
        vals = res_df.loc[res_df['class'] == cls, col]
        ax.hist(vals, bins=30, alpha=0.55, label=cls)
    ax.set_xlabel(label)
    ax.set_ylabel('Image count')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

### 2. Patches per image distribution

In [ ]:
# Count patches per parent image (using the capped train samples)
patches_per_img = Counter(path for path, _, _ in globe_train_patches.samples)
ppi_series = pd.Series(list(patches_per_img.values()), name='patches_per_image')

print('Patches per image (train, after filter + cap):')
print(ppi_series.describe().round(2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(ppi_series, bins=range(1, ppi_series.max() + 2), edgecolor='white', linewidth=0.4)
ax.set_xlabel('Patches per parent image')
ax.set_ylabel('Image count')
ax.set_title('Distribution of patch count per source image (train)')
plt.tight_layout()
plt.show()

### 3. Cloud-filter confidence distribution per class

In [ ]:
# Build per-class confidence arrays from train_confs
conf_by_class = {cls: [] for cls in class_names}
for (path, label_idx, box), conf in train_confs:
    if label_idx < len(class_names):
        conf_by_class[class_names[label_idx]].append(conf)

# Only the 5 GLOBE classes (GCD-only classes aren't filtered)
globe_class_names = [c for c in class_names if c not in GCD_ONLY_CLASSES]

fig, axes = plt.subplots(1, len(globe_class_names),
                         figsize=(len(globe_class_names) * 3.2, 4), sharey=False)
fig.suptitle(f'Cloud-filter confidence distribution (all patches before threshold={CLOUD_THRESHOLD})',
             fontsize=11)

for ax, cls in zip(axes, globe_class_names):
    vals = conf_by_class[cls]
    ax.hist(vals, bins=40, color='steelblue', edgecolor='none')
    ax.axvline(CLOUD_THRESHOLD, color='red', linestyle='--', linewidth=1.2, label=f'thr={CLOUD_THRESHOLD}')
    ax.set_title(cls, fontsize=9)
    ax.set_xlabel('cloud conf')
    if ax is axes[0]:
        ax.set_ylabel('patch count')
    kept_pct = 100 * sum(v >= CLOUD_THRESHOLD for v in vals) / len(vals) if vals else 0
    ax.set_title(f'{cls}\n{kept_pct:.0f}% kept', fontsize=8)

plt.tight_layout()
plt.show()

### 4. Per-class pixel statistics

In [ ]:
# Sample up to N_SAMPLE patches per class for pixel stats
N_SAMPLE = 300
_rng_stat = _random.Random(0)

# Rebuild by_class on the final capped samples
by_class_final = {}
for sample in globe_train_patches.samples:
    by_class_final.setdefault(sample[1], []).append(sample)

stats_records = []

for cls_idx in sorted(by_class_final):
    cls_name = class_names[cls_idx]
    if cls_name in GCD_ONLY_CLASSES:
        continue
    pool   = by_class_final[cls_idx]
    chosen = _rng_stat.sample(pool, min(N_SAMPLE, len(pool)))
    arrays = []
    for path, _, box in chosen:
        try:
            img  = Image.open(path).convert('RGB')
            crop = np.array(img.crop(box).resize((OUTPUT_SIZE, OUTPUT_SIZE), Image.BILINEAR),
                            dtype=np.float32) / 255.0
            arrays.append(crop)
        except Exception:
            pass
    if not arrays:
        continue
    stack = np.stack(arrays)  # (N, H, W, 3)
    for ch_idx, ch_name in enumerate(['R', 'G', 'B']):
        ch = stack[..., ch_idx]
        stats_records.append({
            'class':   cls_name,
            'channel': ch_name,
            'mean':    float(ch.mean()),
            'std':     float(ch.std()),
            'p10':     float(np.percentile(ch, 10)),
            'p50':     float(np.percentile(ch, 50)),
            'p90':     float(np.percentile(ch, 90)),
        })

stats_df = pd.DataFrame(stats_records)
print(stats_df.to_string(index=False))

In [ ]:
# Bar chart: mean pixel value per channel per class
pivot = stats_df.pivot_table(index='class', columns='channel', values='mean')
err   = stats_df.pivot_table(index='class', columns='channel', values='std')

fig, ax = plt.subplots(figsize=(10, 4))
x       = np.arange(len(pivot))
width   = 0.25
colors  = {'R': '#e05252', 'G': '#52a852', 'B': '#5285e0'}

for i, ch in enumerate(['R', 'G', 'B']):
    ax.bar(x + (i - 1) * width, pivot[ch], width,
           yerr=err[ch], capsize=3, label=ch,
           color=colors[ch], alpha=0.85, ecolor='grey')

ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Mean pixel value (0–1)')
ax.set_title('Per-class mean RGB value ± std (sampled patches)')
ax.legend()
plt.tight_layout()
plt.show()

### 5. Mean patch per class

In [ ]:
# Compute and display the pixel-mean of sampled patches for each class
N_MEAN = 500
_rng_mean = _random.Random(1)

mean_patches = {}  # class_name -> (H, W, 3) float32

for cls_idx in sorted(by_class_final):
    cls_name = class_names[cls_idx]
    if cls_name in GCD_ONLY_CLASSES:
        continue
    pool   = by_class_final[cls_idx]
    chosen = _rng_mean.sample(pool, min(N_MEAN, len(pool)))
    arrays = []
    for path, _, box in chosen:
        try:
            img  = Image.open(path).convert('RGB')
            crop = np.array(img.crop(box).resize((OUTPUT_SIZE, OUTPUT_SIZE), Image.BILINEAR),
                            dtype=np.float32) / 255.0
            arrays.append(crop)
        except Exception:
            pass
    if arrays:
        mean_patches[cls_name] = np.mean(np.stack(arrays), axis=0)

n = len(mean_patches)
fig, axes = plt.subplots(1, n, figsize=(n * 3, 3.2))
fig.suptitle(f'Mean patch per class  (n ≤ {N_MEAN} samples)', fontsize=11)

for ax, (cls_name, img_arr) in zip(axes, mean_patches.items()):
    ax.imshow(np.clip(img_arr, 0, 1))
    ax.set_title(cls_name, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

### 6. Sample patch grid

In [ ]:
PATCHES_PER_CLASS = 8
_rng_vis = _random.Random(0)

globe_only = {k: v for k, v in by_class_final.items()
              if class_names[k] not in GCD_ONLY_CLASSES}

n_rows = len(globe_only)
n_cols = PATCHES_PER_CLASS

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.0, n_rows * 2.0))
fig.suptitle(
    f'Sample GLOBE patches  (cloud filter ≥{CLOUD_THRESHOLD}  •  cap {MAX_PATCHES_PER_CLASS:,}/class)',
    fontsize=11, y=1.01
)

for row_idx, cls_idx in enumerate(sorted(globe_only)):
    pool   = globe_only[cls_idx]
    chosen = _rng_vis.sample(pool, min(PATCHES_PER_CLASS, len(pool)))
    for col_idx, (path, label, box) in enumerate(chosen):
        try:
            img  = Image.open(path).convert('RGB')
            crop = img.crop(box).resize((OUTPUT_SIZE, OUTPUT_SIZE), Image.BILINEAR)
        except Exception:
            crop = Image.new('RGB', (OUTPUT_SIZE, OUTPUT_SIZE))
        ax = axes[row_idx][col_idx]
        ax.imshow(crop)
        ax.axis('off')
        if col_idx == 0:
            ax.set_title(class_names[cls_idx], loc='left', fontsize=8,
                         pad=3, fontweight='bold')

plt.tight_layout()
plt.show()

### 7. PCA on patch pixel vectors

In [ ]:
from sklearn.decomposition import PCA

N_PCA   = 200   # patches per class
_rng_pca = _random.Random(2)

X_pca, y_pca = [], []

for cls_idx in sorted(by_class_final):
    cls_name = class_names[cls_idx]
    if cls_name in GCD_ONLY_CLASSES:
        continue
    pool   = by_class_final[cls_idx]
    chosen = _rng_pca.sample(pool, min(N_PCA, len(pool)))
    for path, label, box in chosen:
        try:
            img  = Image.open(path).convert('RGB')
            crop = np.array(img.crop(box).resize((64, 64), Image.BILINEAR),
                            dtype=np.float32).ravel() / 255.0
            X_pca.append(crop)
            y_pca.append(cls_name)
        except Exception:
            pass

X_pca = np.array(X_pca)
print(f'PCA input: {X_pca.shape}  ({len(set(y_pca))} classes)')

pca   = PCA(n_components=2, random_state=0)
Z_pca = pca.fit_transform(X_pca)
print(f'Explained variance ratio: PC1={pca.explained_variance_ratio_[0]:.3f}  '
      f'PC2={pca.explained_variance_ratio_[1]:.3f}')

unique_cls = sorted(set(y_pca))
cmap = plt.get_cmap('tab10')
colors_map = {cls: cmap(i) for i, cls in enumerate(unique_cls)}

fig, ax = plt.subplots(figsize=(8, 6))
for cls in unique_cls:
    mask = np.array(y_pca) == cls
    ax.scatter(Z_pca[mask, 0], Z_pca[mask, 1],
               s=8, alpha=0.5, label=cls, color=colors_map[cls])
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA of flattened 64×64 patches  (GLOBE classes only)')
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout()
plt.show()

### 8. Pixel intensity histograms per class

In [ ]:
# Grayscale intensity distribution per class — useful for seeing brightness differences
N_HIST = 300
_rng_hist = _random.Random(3)

fig, axes = plt.subplots(1, len(globe_only), figsize=(len(globe_only) * 3.2, 4), sharey=True)
fig.suptitle('Grayscale intensity distribution per class (sampled patches)', fontsize=11)

for ax, cls_idx in zip(axes, sorted(globe_only)):
    cls_name = class_names[cls_idx]
    pool     = by_class_final[cls_idx]
    chosen   = _rng_hist.sample(pool, min(N_HIST, len(pool)))
    pixels   = []
    for path, _, box in chosen:
        try:
            img  = Image.open(path).convert('L')  # grayscale
            crop = np.array(img.crop(box).resize((OUTPUT_SIZE, OUTPUT_SIZE), Image.BILINEAR),
                            dtype=np.float32).ravel() / 255.0
            pixels.append(crop)
        except Exception:
            pass
    if pixels:
        flat = np.concatenate(pixels)
        ax.hist(flat, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
        ax.axvline(flat.mean(), color='red', linewidth=1.2, label=f'μ={flat.mean():.2f}')
        ax.legend(fontsize=8)
    ax.set_title(cls_name, fontsize=9)
    ax.set_xlabel('Intensity (0–1)')
    if ax is axes[0]:
        ax.set_ylabel('Pixel count')

plt.tight_layout()
plt.show()